In [1]:
# bunzip2 -c < 0317.raw.bz2 | gzip -c > 0317.raw.gz

In [2]:
import warnings
warnings.filterwarnings('ignore')


from glob import glob
from pathlib import Path
import os
import datetime
import IMProToo   # pip install IMProToo
import gzip    # conda install -c conda-forge gzip 


In [3]:
version = IMProToo.__version__

In [4]:
DATA_DIR = "/scratch/franzihe/"
INPUT_DATA_DIR = os.path.join(DATA_DIR, 'input')
OUTPUT_DATA_DIR = os.path.join(DATA_DIR, 'output')


In [5]:
# make output directory to put in maahn processed data
try:
    new_dir = Path(os.path.join(OUTPUT_DATA_DIR, 'MRR_islas/maahn'))
    new_dir.mkdir(parents=True, exist_ok=True)
except OSError:
    pass

In [6]:
site = "ALOMAR" #"ANX"
skipExisting = True

if site == "ALOMAR":
    file_dir = sorted(glob(os.path.join(INPUT_DATA_DIR, 'alomar_mrr_raw/*/*')))
elif site == "ANX":
    file_dir = sorted(glob(os.path.join(INPUT_DATA_DIR, 'MRR_islas/*raw*/*/*')))

In [7]:
# for nfile in :
for nfile in file_dir:
    # get the timestamp
    if nfile.split('.')[-1] == 'gz':
        f = gzip.open(nfile, 'rt')
    else:
        f = open(nfile, 'rb')

# nfile = sorted(glob(os.path.join(INPUT_DATA_DIR, 'MRR_islas/*raw*/*/*')))[1]
# f = gzip.open(nfile, 'rt')

    # Sometimes the first MRR timestamps are from the day before, so we cannot take the first date we found. get list of line breaks
    line_offset = []
    offset = 0
    for line in f:
        line_offset.append(offset)
        offset += len(line)
    f.seek(0)
    # Now, to skip 20% of the file
    f.seek(line_offset[len(line_offset)//5])

    # find the date
    try:
        while True:
            string = str(f.readline())
            string
            
            if not string:
                print('nothing')
                break

            if string[:2] == "T:":
                timestamp = datetime.datetime.strptime(string[2:14], "%y%m%d%H%M%S").strftime("%Y%m%d")
                break
            elif string[:4] == "MRR ":
                timestamp = datetime.datetime.strptime(string[4:16], "%y%m%d%H%M%S").strftime("%Y%m%d")
                break

    finally:
        f.close()


    if timestamp is None:
        print("did not find MRR timestamp in %s, Skipping" %nfile)
        continue

    file_out = "{}_{}_{}_{}.nc".format(os.path.join(OUTPUT_DATA_DIR, 'MRR_islas/maahn/mrr_improtoo'), version, site, timestamp)

    if skipExisting and (os.path.isfile(file_out) or os.path.isfile(file_out+".gz")):
        print("NetCDF file aready exists, skipping: ", timestamp, nfile, file_out)
        continue
    print(timestamp, nfile, file_out)
    
    # load raw data from file
    print("reading...", nfile)
    try:
        rawData = IMProToo.mrrRawData(nfile)
    except:
        print("could not read data")
        continue

    try:
        # convert rawData object
        processedSpec = IMProToo.MrrZe(rawData)
        # average rawData to 60s
        processedSpec.averageSpectra(60)

        # creator attribute of netCDF file
        # all settings (e.g. creator attribute of netCDF file, dealiasing) are available in the 'processedSpec.co' dictionary 
        # and must be set before calculating Ze etc. See the source code for a description of the settings.
        processedSpec.co["ncCreator"] = "F. Hellmuth"
        processedSpec.co["ncDescription"] = "MRR data processed with IMProToo"
        processedSpec.co["ncLocation"] = "Andenes"
        processedSpec.co["ncInstitution"] = "Dept. of Geoscience, University of Oslo"

        # calculate Ze and other moments
        processedSpec.rawToSnow()

        # write all variables to netCDF file.
        print("writing...", file_out)
        processedSpec.writeNetCDF(file_out, ncForm="NETCDF4_CLASSIC")
    except Exception as error:
        print(str(error))
        print("could not process data")
        continue

NetCDF file aready exists, skipping:  20210301 /scratch/franzihe/input/alomar_mrr_raw/202103/0301.raw.gz /scratch/franzihe/output/MRR_islas/maahn/mrr_improtoo_0.107_ALOMAR_20210301.nc
20210302 /scratch/franzihe/input/alomar_mrr_raw/202103/0302.raw.gz /scratch/franzihe/output/MRR_islas/maahn/mrr_improtoo_0.107_ALOMAR_20210302.nc
reading... /scratch/franzihe/input/alomar_mrr_raw/202103/0302.raw.gz
1 of 1: /scratch/franzihe/input/alomar_mrr_raw/202103/0302.raw.gz


/uio/kant/geo-metos-u1/franzihe/miniconda3/envs/retrieval/lib/python3.9/site-packages/IMProToo/core.py:2584: UserWarning: Warning: some lines without timestamp
  warnings.warn("Warning: some lines without timestamp")
/uio/kant/geo-metos-u1/franzihe/miniconda3/envs/retrieval/lib/python3.9/site-packages/IMProToo/core.py:2667: UserWarning: ? Line not recognized:

  warnings.warn("? Line not recognized:" + dataLine)
/uio/kant/geo-metos-u1/franzihe/miniconda3/envs/retrieval/lib/python3.9/site-packages/IMProToo/core.py:2667: UserWarning: ? Line not recognized:

  warnings.warn("? Line not recognized:" + dataLine)
/uio/kant/geo-metos-u1/franzihe/miniconda3/envs/retrieval/lib/python3.9/site-packages/IMProToo/core.py:2667: UserWarning: ? Line not recognized:

  warnings.warn("? Line not recognized:" + dataLine)
/uio/kant/geo-metos-u1/franzihe/miniconda3/envs/retrieval/lib/python3.9/site-packages/IMProToo/core.py:2667: UserWarning: ? Line not recognized:

  warnings.warn("? Line not recognized:"

No Data at 2021-03-02 07:04:00
No Data at 2021-03-02 07:05:00
No Data at 2021-03-02 07:06:00
No Data at 2021-03-02 07:07:00
No Data at 2021-03-02 07:08:00
No Data at 2021-03-02 07:09:00
No Data at 2021-03-02 07:10:00
No Data at 2021-03-02 07:11:00
No Data at 2021-03-02 07:12:00
No Data at 2021-03-02 07:23:00
No Data at 2021-03-02 07:24:00
No Data at 2021-03-02 07:25:00
No Data at 2021-03-02 07:26:00
No Data at 2021-03-02 07:27:00
No Data at 2021-03-02 07:28:00
No Data at 2021-03-02 07:29:00
No Data at 2021-03-02 07:30:00
No Data at 2021-03-02 07:31:00
No Data at 2021-03-02 07:32:00
No Data at 2021-03-02 07:33:00
No Data at 2021-03-02 07:34:00
No Data at 2021-03-02 07:35:00
No Data at 2021-03-02 07:36:00
No Data at 2021-03-02 07:37:00
No Data at 2021-03-02 07:38:00
No Data at 2021-03-02 07:39:00
No Data at 2021-03-02 07:40:00
No Data at 2021-03-02 07:41:00
No Data at 2021-03-02 07:42:00
No Data at 2021-03-02 07:43:00
No Data at 2021-03-02 07:44:00
No Data at 2021-03-02 07:45:00
No Data 

/uio/kant/geo-metos-u1/franzihe/miniconda3/envs/retrieval/lib/python3.9/site-packages/IMProToo/core.py:2584: UserWarning: Warning: some lines without timestamp
  warnings.warn("Warning: some lines without timestamp")
/uio/kant/geo-metos-u1/franzihe/miniconda3/envs/retrieval/lib/python3.9/site-packages/IMProToo/core.py:2667: UserWarning: ? Line not recognized:

  warnings.warn("? Line not recognized:" + dataLine)
/uio/kant/geo-metos-u1/franzihe/miniconda3/envs/retrieval/lib/python3.9/site-packages/IMProToo/core.py:2667: UserWarning: ? Line not recognized:

  warnings.warn("? Line not recognized:" + dataLine)
/uio/kant/geo-metos-u1/franzihe/miniconda3/envs/retrieval/lib/python3.9/site-packages/IMProToo/core.py:2667: UserWarning: ? Line not recognized:

  warnings.warn("? Line not recognized:" + dataLine)
/uio/kant/geo-metos-u1/franzihe/miniconda3/envs/retrieval/lib/python3.9/site-packages/IMProToo/core.py:2667: UserWarning: ? Line not recognized:

  warnings.warn("? Line not recognized:"

No Data at 2021-03-03 07:04:00
No Data at 2021-03-03 07:05:00
No Data at 2021-03-03 07:06:00
No Data at 2021-03-03 07:07:00
No Data at 2021-03-03 07:08:00
No Data at 2021-03-03 07:09:00
No Data at 2021-03-03 07:10:00
No Data at 2021-03-03 07:11:00
No Data at 2021-03-03 07:12:00
No Data at 2021-03-03 07:23:00
No Data at 2021-03-03 07:24:00
No Data at 2021-03-03 07:25:00
No Data at 2021-03-03 07:26:00
No Data at 2021-03-03 07:27:00
No Data at 2021-03-03 07:28:00
No Data at 2021-03-03 07:29:00
No Data at 2021-03-03 07:30:00
No Data at 2021-03-03 07:31:00
No Data at 2021-03-03 07:32:00
No Data at 2021-03-03 07:33:00
No Data at 2021-03-03 07:34:00
No Data at 2021-03-03 07:35:00
No Data at 2021-03-03 07:36:00
No Data at 2021-03-03 07:37:00
No Data at 2021-03-03 07:38:00
No Data at 2021-03-03 07:39:00
No Data at 2021-03-03 07:40:00
No Data at 2021-03-03 07:41:00
No Data at 2021-03-03 07:42:00
No Data at 2021-03-03 07:43:00
No Data at 2021-03-03 07:44:00
No Data at 2021-03-03 07:45:00
No Data 

...

/uio/kant/geo-metos-u1/franzihe/miniconda3/envs/retrieval/lib/python3.9/site-packages/IMProToo/core.py:2584: UserWarning: Warning: some lines without timestamp
  warnings.warn("Warning: some lines without timestamp")
/uio/kant/geo-metos-u1/franzihe/miniconda3/envs/retrieval/lib/python3.9/site-packages/IMProToo/core.py:2667: UserWarning: ? Line not recognized:

  warnings.warn("? Line not recognized:" + dataLine)
/uio/kant/geo-metos-u1/franzihe/miniconda3/envs/retrieval/lib/python3.9/site-packages/IMProToo/core.py:2667: UserWarning: ? Line not recognized:

  warnings.warn("? Line not recognized:" + dataLine)
/uio/kant/geo-metos-u1/franzihe/miniconda3/envs/retrieval/lib/python3.9/site-packages/IMProToo/core.py:2667: UserWarning: ? Line not recognized:

  warnings.warn("? Line not recognized:" + dataLine)
/uio/kant/geo-metos-u1/franzihe/miniconda3/envs/retrieval/lib/python3.9/site-packages/IMProToo/core.py:2667: UserWarning: ? Line not recognized:

  warnings.warn("? Line not recognized:"

No Data at 2021-03-08 04:53:00
No Data at 2021-03-08 04:54:00
No Data at 2021-03-08 04:55:00
No Data at 2021-03-08 04:56:00
No Data at 2021-03-08 04:57:00
No Data at 2021-03-08 04:58:00
No Data at 2021-03-08 04:59:00
No Data at 2021-03-08 05:00:00
No Data at 2021-03-08 05:01:00
No Data at 2021-03-08 05:02:00
No Data at 2021-03-08 05:03:00
No Data at 2021-03-08 05:04:00
No Data at 2021-03-08 05:05:00
No Data at 2021-03-08 05:06:00
No Data at 2021-03-08 05:07:00
No Data at 2021-03-08 05:08:00
No Data at 2021-03-08 05:09:00
No Data at 2021-03-08 05:10:00
No Data at 2021-03-08 05:11:00
No Data at 2021-03-08 05:12:00
No Data at 2021-03-08 05:13:00
No Data at 2021-03-08 05:14:00
No Data at 2021-03-08 05:15:00
No Data at 2021-03-08 05:16:00
No Data at 2021-03-08 05:17:00
No Data at 2021-03-08 05:18:00
No Data at 2021-03-08 05:19:00
No Data at 2021-03-08 05:20:00
No Data at 2021-03-08 05:21:00
No Data at 2021-03-08 05:22:00
No Data at 2021-03-08 05:23:00
No Data at 2021-03-08 05:24:00
No Data 

/uio/kant/geo-metos-u1/franzihe/miniconda3/envs/retrieval/lib/python3.9/site-packages/IMProToo/core.py:2584: UserWarning: Warning: some lines without timestamp
  warnings.warn("Warning: some lines without timestamp")
/uio/kant/geo-metos-u1/franzihe/miniconda3/envs/retrieval/lib/python3.9/site-packages/IMProToo/core.py:2667: UserWarning: ? Line not recognized:

  warnings.warn("? Line not recognized:" + dataLine)
/uio/kant/geo-metos-u1/franzihe/miniconda3/envs/retrieval/lib/python3.9/site-packages/IMProToo/core.py:2667: UserWarning: ? Line not recognized:

  warnings.warn("? Line not recognized:" + dataLine)
/uio/kant/geo-metos-u1/franzihe/miniconda3/envs/retrieval/lib/python3.9/site-packages/IMProToo/core.py:2667: UserWarning: ? Line not recognized:

  warnings.warn("? Line not recognized:" + dataLine)
/uio/kant/geo-metos-u1/franzihe/miniconda3/envs/retrieval/lib/python3.9/site-packages/IMProToo/core.py:2667: UserWarning: ? Line not recognized:

  warnings.warn("? Line not recognized:"

No Data at 2021-03-09 06:58:00
No Data at 2021-03-09 06:59:00
No Data at 2021-03-09 07:15:00
No Data at 2021-03-09 07:16:00
No Data at 2021-03-09 07:17:00
No Data at 2021-03-09 07:18:00
No Data at 2021-03-09 07:19:00
No Data at 2021-03-09 07:20:00
No Data at 2021-03-09 07:21:00
No Data at 2021-03-09 07:22:00
No Data at 2021-03-09 07:23:00
No Data at 2021-03-09 07:24:00
No Data at 2021-03-09 07:25:00
No Data at 2021-03-09 07:26:00
No Data at 2021-03-09 07:27:00
No Data at 2021-03-09 07:28:00
No Data at 2021-03-09 07:29:00
No Data at 2021-03-09 07:30:00
No Data at 2021-03-09 07:31:00
No Data at 2021-03-09 07:32:00
No Data at 2021-03-09 07:33:00
No Data at 2021-03-09 07:38:00
No Data at 2021-03-09 07:39:00
No Data at 2021-03-09 07:40:00
No Data at 2021-03-09 07:41:00
No Data at 2021-03-09 07:42:00
No Data at 2021-03-09 07:43:00
No Data at 2021-03-09 07:44:00
No Data at 2021-03-09 07:45:00
No Data at 2021-03-09 07:46:00
No Data at 2021-03-09 07:47:00
No Data at 2021-03-09 07:48:00
No Data 

UnicodeDecodeError: 'utf-8' codec can't decode byte 0xa3 in position 7703: invalid start byte

In [ ]:
f

<_io.TextIOWrapper name='/scratch/franzihe/input/alomar_mrr_raw/202103/0310.raw.gz' encoding='UTF-8'>

The history saving thread hit an unexpected error (OperationalError('disk I/O error')).History will not be written to the database.


In [ ]:
# rsync -av --progress --exclude "*.bz2" login.nird.sigma2.no:/projects/NS9600K/data/islas2022/MRR_islas /scratch/franzihe/input

# rsync -av --progress login.nird.sigma2.no:/projects/NS9600K/data/islas/alomar_mrr_raw /scratch/franzihe/input/